Sección 1: Imports y configuración globalEsta celda contiene la importación de todas las librerías necesarias para el desarrollo de los experimentos de detección de objetos, incluyendo los componentes de torchvision para los modelos Faster R-CNN y RetinaNet, así como torchmetrics para el cálculo de mAP. También se configuran las semillas de aleatoriedad para garantizar la reproducibilidad y se definen las rutas globales y los hiperparámetros de entrenamiento.  

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F  # Reservado estándar para funciones de NN
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset
from torchmetrics.detection import MeanAveragePrecision
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.transforms import functional as TF  # Cambiado a TF para evitar colisiones
import yaml

# --- Configuración de Rutas ---
ROOT = Path("..")
DATA_ROOT = ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed" / "dataset_no_background"

SPLIT_CSV = {
    "train": DATA_ROOT / "train.csv",
    "valid": DATA_ROOT / "val.csv",
    "test": DATA_ROOT / "test.csv",
}

# --- Hiperparámetros Globales ---
IMAGE_SIZE = 224
BATCH_SIZE = 8
NUM_EPOCHS = 40
NUM_WORKERS = 4
SEED = 42

# --- Fijar Semilla para Reproducibilidad ---
def seed_everything(seed=SEED):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything()

# --- Configuración de Dispositivo ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams["figure.figsize"] = (12, 6)

# --- Cargar nombres de clases desde data.yaml ---
with open(PROCESSED_ROOT / "data.yaml", "r", encoding="utf-8") as f:
    yaml_data = yaml.safe_load(f)
CLASS_NAMES = yaml_data.get("names", [])
NUM_CLASSES = len(CLASS_NAMES)

print(f"Dispositivo: {device} | Clases detectadas: {CLASS_NAMES} ({NUM_CLASSES})")

Sección 2: Dataset y DataLoaders para detección
En esta sección se definen las funciones de utilidad reutilizadas del pipeline de preparación de datos, incluyendo la lectura de anotaciones YOLO normalizadas. Se implementa la transformación de coordenadas yolo_to_xyxy para mapear el formato (x_center, y_center, width, height) normalizado al formato Pascal VOC absoluto [x1, y1, x2, y2] requerido por los modelos de torchvision. Asimismo, la clase CandlestickDetectionDataset incorpora un parámetro label_offset para alternar correctamente el índice de las etiquetas según el modelo utilizado (1 a 6 para Faster R-CNN guardando el 0 para el fondo, y 0 a 5 para RetinaNet).

In [ ]:
def convert_to_rgb(img: Image.Image) -> Image.Image:
    """Asegura la conversión de imágenes RGBA descomponiendo contra un fondo negro."""
    if img.mode == "RGBA":
        background = Image.new("RGB", img.size, (0, 0, 0))
        background.paste(img, mask=img.split()[3])
        return background
    return img.convert("RGB") if img.mode != "RGB" else img

def read_yolo_annotations(label_path: Path) -> list:
    """Lee anotaciones en formato YOLO de archivos de texto."""
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) >= 5:
            try:
                boxes.append((
                    int(parts[0]),
                    float(parts[1]),
                    float(parts[2]),
                    float(parts[3]),
                    float(parts[4]),
                ))
            except ValueError:
                continue
    return boxes

def yolo_to_xyxy(xc, yc, w, h, img_w=IMAGE_SIZE, img_h=IMAGE_SIZE):
    """Transforma coordenadas YOLO [0,1] a Pascal VOC absolutizadas [x1, y1, x2, y2]."""
    x1 = (xc - w / 2) * img_w
    y1 = (yc - h / 2) * img_h
    x2 = (xc + w / 2) * img_w
    y2 = (yc + h / 2) * img_h
    return [x1, y1, x2, y2]

def detection_collate_fn(batch):
    """Collate especial para evitar colapsos por cantidad variable de cajas por imagen."""
    images, targets = zip(*batch)
    return torch.stack(images, dim=0), list(targets)

class CandlestickDetectionDataset(Dataset):
    """Dataset personalizado para detección de patrones de velas japonesas."""
    def __init__(self, csv_path: Path, dataset_root: Path, label_offset=0, transform=None):
        self.df = pd.read_csv(csv_path)
        self.dataset_root = dataset_root
        self.label_offset = label_offset
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image_path = self.dataset_root / row["image"]
        label_path = self.dataset_root / row["label"]

        img_pil = Image.open(image_path)
        img_pil = convert_to_rgb(img_pil)
        img_pil = img_pil.resize((IMAGE_SIZE, IMAGE_SIZE))

        annotations = read_yolo_annotations(label_path)
        boxes_list = []
        labels_list = []

        for cls_id, xc, yc, bw, bh in annotations:
            bbox = yolo_to_xyxy(xc, yc, bw, bh, IMAGE_SIZE, IMAGE_SIZE)
            boxes_list.append(bbox)
            labels_list.append(cls_id + self.label_offset)

        if boxes_list:
            boxes = torch.tensor(boxes_list, dtype=torch.float32)
            labels = torch.tensor(labels_list, dtype=torch.long)
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.long)

        img_tensor = TF.to_tensor(img_pil) # Actualizado de F a TF
        if self.transform:
            img_tensor = self.transform(img_tensor)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
        }
        return img_tensor, target

Sección 3: Análisis del desbalance de clases
Esta celda realiza un análisis explícito de la frecuencia de instancias presentes dentro del conjunto de entrenamiento. Se visualiza la distribución de las 6 clases para justificar la necesidad del uso de técnicas de balanceo, calculando los pesos frecuenciales inversos globales y extrayendo un vector de ponderación por muestra (sample_weights) listo para el sobremuestreo mediante WeightedRandomSampler.

In [ ]:
# Contar instancias reales por cada clase en el split de entrenamiento
train_df = pd.read_csv(SPLIT_CSV["train"])
class_counts = Counter()

for _, row in train_df.iterrows():
    annotations = read_yolo_annotations(PROCESSED_ROOT / row["label"])
    for cls_id, *_ in annotations:
        class_counts[cls_id] += 1

for i in range(NUM_CLASSES):
    if i not in class_counts:
        class_counts[i] = 0

# Graficar la distribución de clases
classes = [CLASS_NAMES[i] for i in range(NUM_CLASSES)]
counts = [class_counts[i] for i in range(NUM_CLASSES)]

plt.figure()
plt.bar(classes, counts, color="teal")
plt.title("Distribución de Instancias por Clase en Train Set")
plt.xlabel("Patrón de Vela")
plt.ylabel("Cantidad de Cajas")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.3)
plt.show()

# Cálculo de pesos por frecuencia inversa
total_instances = sum(class_counts.values())
class_weights = {}
for i in range(NUM_CLASSES):
    class_weights[i] = (
        total_instances / (NUM_CLASSES * class_counts[i])
        if class_counts[i] > 0
        else 1.0
    )

class_weights_tensor = torch.tensor(
    [class_weights[i] for i in range(NUM_CLASSES)], dtype=torch.float32
).to(device)
print(f"Pesos de Clase (Frecuencia Inversa): {class_weights}")

Sección 4: Definición del modelo y estrategia de fine-tuning
En esta celda se define la función encargada de instanciar la red Faster R-CNN basada en un backbone preentrenado ResNet50-FPN. Modificamos la cabeza predictora (box_predictor) adaptándola al número de clases requeridas (6 patrones + 1 fondo). Se determina aplicar un full fine-tuning entrenando todos los parámetros para adaptar de forma óptima el conocimiento abstracto del dataset COCO a las características geométricas de las velas japonesas.CAMBIAR

In [ ]:
def build_faster_rcnn_model(num_classes=7):
    """Construye un modelo Faster R-CNN con backbone ResNet50 preentrenado."""
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights="COCO_V1",
        box_nms_thresh=0.3
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    for param in model.parameters():
        param.requires_grad = True
    return model

def softmax_focal_loss(inputs, targets, gamma=2.0, alpha=0.25):
    """Focal loss multiclase usando softmax. Reduce a cross-entropy cuando gamma=0."""
    ce = F.cross_entropy(inputs, targets, reduction='none')
    pt = torch.exp(-ce)
    return (alpha * (1 - pt) ** gamma * ce).mean()

def build_retinanet_model(num_classes=6, focal_gamma=2.0, focal_alpha=0.25):
    """Construye un modelo RetinaNet parametrizando los coeficientes de Focal Loss."""
    model = torchvision.models.detection.retinanet_resnet50_fpn(weights="COCO_V1")
    in_channels = model.head.classification_head.conv[0][0].in_channels
    num_anchors = model.head.classification_head.num_anchors
    
    model.head.classification_head = RetinaNetClassificationHead(
        in_channels, num_anchors, num_classes=num_classes
    )
    
    # Ajustar parámetros de Focal Loss (gamma=0 desactiva el efecto focal)
    model.head.classification_head.focal_loss_gamma = focal_gamma
    model.head.classification_head.focal_loss_alpha = focal_alpha
    
    for param in model.parameters():
        param.requires_grad = True
    return model



Sección 5: Loop de entrenamiento genérico
Esta sección define la función maestra para orquestar los procesos de entrenamiento y validación de las redes de detección de objetos. La fase de entrenamiento realiza el forward pass extrayendo y acumulando los diccionarios de pérdidas nativas de torchvision. Por otro lado, la fase de validación conmuta el modelo a modo .eval() y delega el cálculo exhaustivo de la métrica mAP@0.5 a la librería torchmetrics. El loop además guarda el mejor estado de pesos basado en el rendimiento obtenido en validación

In [ ]:
def train_detection_model(
    model,
    train_loader,
    valid_loader,
    optimizer,
    scheduler,
    num_epochs,
    device,
    exp_name,
):
    """Loop de entrenamiento y validación estándar con mAP extendido por clase."""
    model.to(device)
    history = {"train_loss": [], "val_mAP": [], "val_mAP_per_class": []}
    best_map = -1.0

    for epoch in range(1, num_epochs + 1):
        # --- Fase de Entrenamiento ---
        model.train()
        epoch_losses = []

        for images, targets in train_loader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(images, targets)
            losses = sum(loss_for_type for loss_for_type in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()

            epoch_losses.append(losses.item())

        mean_train_loss = np.mean(epoch_losses)
        scheduler.step()

        # --- Fase de Validación ---
        model.eval()
        # Modificación 1: Habilitar class_metrics=True
        metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)

        with torch.no_grad():
            for images, targets in valid_loader:
                images = [img.to(device) for img in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                predictions = model(images)
                metric.update(predictions, targets)

        metrics_results = metric.compute()
        current_map = metrics_results["map_50"].item()
        
        # Modificación 2: Extraer y guardar mAP@0.5 por clase
        per_class = metrics_results["map_per_class"].cpu().numpy()

        history["train_loss"].append(mean_train_loss)
        history["val_mAP"].append(current_map)
        history["val_mAP_per_class"].append(per_class)

        print(
            f"[{exp_name}] Epoch {epoch}/{num_epochs} -> Train Loss: {mean_train_loss:.4f} | Val mAP@0.5: {current_map:.4f}"
        )

        if current_map > best_map:
            best_map = current_map
            checkpoint_path = ROOT / "dev" / f"{exp_name}_best.pth"
            checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), checkpoint_path)

    return history

Sección 6: Experimento 1 — Baseline sin balanceo de clases
Se ejecuta el primer experimento configurando un entrenamiento estándar sin aplicar ninguna técnica paliativa contra el desbalance de clases. El DataLoader utiliza un mezclado estándar (shuffle=True), alimentando un Faster R-CNN convencional para observar el comportamiento base del modelo frente a categorías fuertemente minoritarias.

In [ ]:
print("--- Iniciando Experimento 1: Baseline Faster R-CNN ---")

train_dataset_exp1 = CandlestickDetectionDataset(SPLIT_CSV["train"], PROCESSED_ROOT, label_offset=1)
valid_dataset_exp1 = CandlestickDetectionDataset(SPLIT_CSV["valid"], PROCESSED_ROOT, label_offset=1)

train_loader_exp1 = DataLoader(train_dataset_exp1, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)
valid_loader_exp1 = DataLoader(valid_dataset_exp1, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)

model_exp1 = build_faster_rcnn_model(num_classes=7)
optimizer_exp1 = torch.optim.SGD(model_exp1.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler_exp1 = torch.optim.lr_scheduler.StepLR(optimizer_exp1, step_size=10, gamma=0.5)

history_exp1 = train_detection_model(
    model=model_exp1,
    train_loader=train_loader_exp1,
    valid_loader=valid_loader_exp1,
    optimizer=optimizer_exp1,
    scheduler=scheduler_exp1,
    num_epochs=NUM_EPOCHS,
    device=device,
    exp_name="exp1_baseline",
)

Sección 7: Experimento 2 — WeightedRandomSampler (sobremuestreo)
En este segundo experimento intervenimos a nivel de la carga de datos empleando un sobremuestreo controlado. Desactivamos la propiedad shuffle nativa y delegamos la selección de muestras a un WeightedRandomSampler cargado con los pesos previamente extraídos, forzando una mayor paridad en la aparición de las clases raras dentro de los minibatches de entrenamiento

In [ ]:
print("--- Iniciando Experimento 2: Faster R-CNN con Focal Loss ---")

train_dataset_exp2 = CandlestickDetectionDataset(SPLIT_CSV["train"], PROCESSED_ROOT, label_offset=1)
valid_dataset_exp2 = CandlestickDetectionDataset(SPLIT_CSV["valid"], PROCESSED_ROOT, label_offset=1)

train_loader_exp2 = DataLoader(train_dataset_exp2, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)
valid_loader_exp2 = DataLoader(valid_dataset_exp2, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)

model_exp2 = build_faster_rcnn_model(num_classes=7)
optimizer_exp2 = torch.optim.SGD(model_exp2.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler_exp2 = torch.optim.lr_scheduler.StepLR(optimizer_exp2, step_size=10, gamma=0.5)

# Configuración del Monkey-patch puntual
import torchvision.models.detection.roi_heads as _rh
_original_loss = _rh.fastrcnn_loss

def _focal_loss_fn(class_logits, box_regression, labels, regression_targets):
    labels_cat = torch.cat(labels)
    regression_targets_cat = torch.cat(regression_targets)
    classification_loss = softmax_focal_loss(class_logits, labels_cat, gamma=2.0, alpha=0.25)
    
    sampled_pos = torch.where(labels_cat > 0)[0]
    labels_pos = labels_cat[sampled_pos]
    N = class_logits.shape[0]
    
    box_regression = box_regression.reshape(N, -1, 4)
    box_loss = F.smooth_l1_loss(
        box_regression[sampled_pos, labels_pos],
        regression_targets_cat[sampled_pos], beta=1/9, reduction='sum'
    ) / max(labels_cat.numel(), 1)
    return classification_loss, box_loss

# Inyectar función focal
_rh.fastrcnn_loss = _focal_loss_fn

history_exp2 = train_detection_model(
    model=model_exp2,
    train_loader=train_loader_exp2,
    valid_loader=valid_loader_exp2,
    optimizer=optimizer_exp2,
    scheduler=scheduler_exp2,
    num_epochs=NUM_EPOCHS,
    device=device,
    exp_name="exp2_focal",
)

# Restaurar función original
_rh.fastrcnn_loss = _original_loss

Sección 8: Experimento 3


In [ ]:
print("--- Iniciando Experimento 3: RetinaNet Baseline ---")

train_dataset_exp3 = CandlestickDetectionDataset(SPLIT_CSV["train"], PROCESSED_ROOT, label_offset=0)
valid_dataset_exp3 = CandlestickDetectionDataset(SPLIT_CSV["valid"], PROCESSED_ROOT, label_offset=0)

train_loader_exp3 = DataLoader(train_dataset_exp3, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)
valid_loader_exp3 = DataLoader(valid_dataset_exp3, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)

# Desactiva focal weighting usando gamma=0.0
model_exp3 = build_retinanet_model(num_classes=6, focal_gamma=0.0, focal_alpha=0.25)
optimizer_exp3 = torch.optim.SGD(model_exp3.parameters(), lr=0.002, momentum=0.9, weight_decay=0.0005)
scheduler_exp3 = torch.optim.lr_scheduler.StepLR(optimizer_exp3, step_size=10, gamma=0.5)

history_exp3 = train_detection_model(
    model=model_exp3,
    train_loader=train_loader_exp3,
    valid_loader=valid_loader_exp3,
    optimizer=optimizer_exp3,
    scheduler=scheduler_exp3,
    num_epochs=NUM_EPOCHS,
    device=device,
    exp_name="exp3_retinanet_baseline",
)

Sección 9: Experimento 4 — RetinaNet con Focal Loss
El tercer experimento cambia la arquitectura de dos etapas por un detector de una sola etapa: RetinaNet. Al evaluar miles de anchors de forma directa, procesa el desbalance mediante la aplicación nativa de la Focal Loss en su cabeza de clasificación. Es importante notar que RetinaNet indexa las clases de manera directa de 0 a 5 sin requerir la reserva del índice cero para el fondo (label_offset=0).

In [ ]:
print("--- Iniciando Experimento 4: RetinaNet con Focal Loss ---")

train_dataset_exp4 = CandlestickDetectionDataset(SPLIT_CSV["train"], PROCESSED_ROOT, label_offset=0)
valid_dataset_exp4 = CandlestickDetectionDataset(SPLIT_CSV["valid"], PROCESSED_ROOT, label_offset=0)

train_loader_exp4 = DataLoader(train_dataset_exp4, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)
valid_loader_exp4 = DataLoader(valid_dataset_exp4, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)

# Activa focal weighting usando gamma=2.0
model_exp4 = build_retinanet_model(num_classes=6, focal_gamma=2.0, focal_alpha=0.25)
optimizer_exp4 = torch.optim.SGD(model_exp4.parameters(), lr=0.002, momentum=0.9, weight_decay=0.0005)
scheduler_exp4 = torch.optim.lr_scheduler.StepLR(optimizer_exp4, step_size=10, gamma=0.5)

history_exp4 = train_detection_model(
    model=model_exp4,
    train_loader=train_loader_exp4,
    valid_loader=valid_loader_exp4,
    optimizer=optimizer_exp4,
    scheduler=scheduler_exp4,
    num_epochs=NUM_EPOCHS,
    device=device,
    exp_name="exp4_retinanet_focal",
)

Sección 10: Tabla comparativa de experimentos
Se consolidan los históricos de los tres entrenamientos ejecutados para generar una tabla comparativa en formato DataFrame. Se evalúan las dinámicas de convergencia visualizando las curvas superpuestas de la función de pérdida (train_loss) y la precisión media promedio (val_mAP) a lo largo del transcurso de las épocas

In [ ]:
# --- 1. Tabla Comparativa ---
data_summary = [
    {
        "Experimento": "Exp 1",
        "Modelo Base": "Faster R-CNN",
        "Estrategia Balanceo": "Ninguna (Baseline)",
        "Final Train Loss": history_exp1["train_loss"][-1],
        "Val mAP@0.5": history_exp1["val_mAP"][-1]
    },
    {
        "Experimento": "Exp 2",
        "Modelo Base": "Faster R-CNN",
        "Estrategia Balanceo": "Focal Loss (Patch)",
        "Final Train Loss": history_exp2["train_loss"][-1],
        "Val mAP@0.5": history_exp2["val_mAP"][-1]
    },
    {
        "Experimento": "Exp 3",
        "Modelo Base": "RetinaNet",
        "Estrategia Balanceo": "Ninguna (Baseline)",
        "Final Train Loss": history_exp3["train_loss"][-1],
        "Val mAP@0.5": history_exp3["val_mAP"][-1]
    },
    {
        "Experimento": "Exp 4",
        "Modelo Base": "RetinaNet",
        "Estrategia Balanceo": "Focal Loss (Nativa)",
        "Final Train Loss": history_exp4["train_loss"][-1],
        "Val mAP@0.5": history_exp4["val_mAP"][-1]
    }
]

df_summary = pd.DataFrame(data_summary)
print(df_summary.to_string(index=False))

# --- 2. Configuración de Gráficos de Línea ---
epochs = range(1, NUM_EPOCHS + 1)
legends = [
    "Exp1: FasterRCNN Baseline", 
    "Exp2: FasterRCNN Focal", 
    "Exp3: RetinaNet Baseline", 
    "Exp4: RetinaNet Focal"
]
histories = [history_exp1, history_exp2, history_exp3, history_exp4]

# Gráfico 1 — Evolución de Train Loss (Fila Propia)
plt.figure(figsize=(12, 5))
for h in histories:
    plt.plot(epochs, h["train_loss"], linewidth=2)
plt.title("Evolución de Train Loss", fontsize=14)
plt.xlabel("Épocas")
plt.ylabel("Loss")
plt.legend(legends)
plt.grid(True, alpha=0.3)
plt.show()

# Gráfico 2 — Evolución de mAP@0.5 General (Fila Propia)
plt.figure(figsize=(12, 5))
for h in histories:
    plt.plot(epochs, h["val_mAP"], linewidth=2)
plt.title("Evolución de mAP@0.5 General", fontsize=14)
plt.xlabel("Épocas")
plt.ylabel("mAP@0.5")
plt.legend(legends)
plt.grid(True, alpha=0.3)
plt.show()

# Gráficos 3 a 8 — mAP@0.5 por clase (Grid 2x3)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i in range(NUM_CLASSES):
    ax = axes[i]
    for h in histories:
        class_curve = [epoch_per_class[i] for epoch_per_class in h["val_mAP_per_class"]]
        ax.plot(epochs, class_curve, linewidth=1.5)
    ax.set_title(f"mAP@0.5 - {CLASS_NAMES[i]}", fontsize=12)
    ax.set_xlabel("Épocas")
    ax.set_ylabel("mAP@0.5")
    ax.legend(legends, fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Sección 11: Evaluación final del modelo elegido (test set)
Tras analizar los resultados comparativos de validación, seleccionamos el mejor experimento (por ejemplo, el modelo del Experimento 2 con WeightedRandomSampler), cargamos sus pesos de disco y ejecutamos un testeo exhaustivo sobre el conjunto independiente test_loader. El reporte desglosa los valores definitivos de mAP@0.5, mAP@0.5:0.95 y la precisión promedio calculada de forma individual para cada uno de los 6 patrones de velas japonesas.

In [ ]:
# Selección automática del mejor experimento basado en el mAP de validación
best_maps = [h["val_mAP"][-1] for h in histories]
winner_idx = np.argmax(best_maps)
winner_exp_name = ["exp1_baseline", "exp2_focal", "exp3_retinanet_baseline", "exp4_retinanet_focal"][winner_idx]

print(f"Ganador del entrenamiento: {legends[winner_idx]} con mAP@0.5 de {best_maps[winner_idx]:.4f}")

# Reconstruir dinámicamente el modelo ganador para evaluación final
if "FasterRCNN" in legends[winner_idx]:
    production_model = build_faster_rcnn_model(num_classes=7)
    test_label_offset = 1
else:
    production_model = build_retinanet_model(num_classes=6)
    test_label_offset = 0

# Cargar pesos guardados del ganador
winner_checkpoint = ROOT / "dev" / f"{winner_exp_name}_best.pth"
production_model.load_state_dict(torch.load(winner_checkpoint, map_location=device))
production_model.to(device)
production_model.eval()

# Instanciar el Test DataLoader con el offset adecuado
test_dataset = CandlestickDetectionDataset(SPLIT_CSV["test"], PROCESSED_ROOT, label_offset=test_label_offset)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=detection_collate_fn)

# Evaluación definitiva en Test Set
test_metric = MeanAveragePrecision(iou_type="bbox", class_metrics=True)
with torch.no_grad():
    for images, targets in test_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        predictions = production_model(images)
        test_metric.update(predictions, targets)

test_results = test_metric.compute()
print("\n=== EVALUACIÓN FINAL EN TEST SET ===")
print(f"mAP@0.5 General: {test_results['map_50'].item():.4f}")

for i, class_name in enumerate(CLASS_NAMES):
    print(f"mAP@0.5 por Clase [{class_name}]: {test_results['map_per_class'][i].item():.4f}")

Sección 12: Análisis de errores
Esta función selecciona muestras aleatorias extraídas directamente del conjunto de test para graficar una cuadrícula visual. Se dibujan las cajas de predicción generadas por la red neuronal (en líneas continuas) y se contrastan con las etiquetas reales del ground truth (en líneas punteadas) para identificar de forma intuitiva la tasa de falsos positivos y falsos negativos

In [ ]:
def visualize_predictions_vs_ground_truth(model, dataset, num_images=8):
    """Muestra una grilla comparativa contrastando inferencias de la red contra etiquetas reales."""
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    axes = axes.flatten()

    indices = random.sample(range(len(dataset)), num_images)

    for idx, sample_idx in enumerate(indices):
        img_tensor, target = dataset[sample_idx]

        # Pasar tensor al dispositivo e inferir
        model.eval()
        with torch.no_grad():
            prediction = model([img_tensor.to(device)])[0]

        # Desnormalizar la imagen tensor para poder graficarla con PIL
        img_np = img_tensor.permute(1, 2, 0).numpy()
        img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)
        img_pil = Image.fromarray(img_np)
        draw = ImageDraw.Draw(img_pil)

        # 1. Dibujar Bounding Boxes Reales (Ground Truth) en azul discontinuo
        for box, lbl in zip(target["boxes"], target["labels"]):
            x1, y1, x2, y2 = box.tolist()
            draw.rectangle([x1, y1, x2, y2], outline="yellow", width=2)
            # Reajustar texto restando el offset para visualización correcta
            cls_idx = lbl.item() - dataset.label_offset
            draw.text((x1 + 3, y1 + 3), f"GT: {CLASS_NAMES[cls_idx]}", fill="yellow")

        # 2. Dibujar Bounding Boxes Predichos en rojo continuo (filtrado por umbral de confianza)
        CONF_THRESHOLD = 0.5
        pred_boxes = prediction["boxes"].cpu()
        pred_labels = prediction["labels"].cpu()
        pred_scores = prediction["scores"].cpu()

        for box, lbl, score in zip(pred_boxes, pred_labels, pred_scores):
            if score.item() >= CONF_THRESHOLD:
                x1, y1, x2, y2 = box.tolist()
                draw.rectangle([x1, y1, x2, y2], outline="cyan", width=2)
                cls_idx = lbl.item() - dataset.label_offset
                draw.text(
                    (x1 + 3, y2 - 12),
                    f"PR: {CLASS_NAMES[cls_idx]} ({score.item():.2f})",
                    fill="cyan",
                )

        axes[idx].imshow(img_pil)
        axes[idx].axis("off")
        axes[idx].set_title(f"Sample Index {sample_idx}", fontsize=9)

    plt.suptitle(
        "Análisis Visual: Ground Truth (Azul) vs Predicciones de la Red (Rojo)",
        fontsize=14,
    )
    plt.tight_layout()
    plt.show()


# Ejecutar la función de diagnóstico de errores
visualize_predictions_vs_ground_truth(production_model, test_dataset, num_images=8)

Sección 13: Guardado del modelo
Finalmente, se exporta el diccionario de estados de pesos del mejor modelo optimizado hacia la ruta unificada requerida por la cátedra (dev/modelo.pth). Asimismo, se genera y anexa una advertencia de control referida a las limitantes de peso de la plataforma GitHub y la configuración del archivo .gitattributes para activar el rastreo mediante Git LFS (Large File Storage).

In [ ]:
# Definición de la ruta final estipulada para producción
production_model_path = ROOT / "dev" / "modelo.pth"
production_model_path.parent.mkdir(parents=True, exist_ok=True)

# Guardado definitivo de los parámetros del modelo para posterior consumo en prod/
torch.save(production_model.state_dict(), production_model_path)
print(f"¡Modelo de producción guardado con éxito en: {production_model_path}!")

# --- Verificación de peso para Git LFS ---
file_size_mb = os.path.getsize(production_model_path) / (1024 * 1024)
print(f"Tamaño estimado del archivo: {file_size_mb:.2f} MB")

if file_size_mb > 100:
    print("\n[ALERTA DE SEGURIDAD - GITHUB]")
    print(
        "El archivo 'modelo.pth' supera los 100 MB de almacenamiento límite."
    )
    print(
        "Por favor, configure Git LFS ejecutando las siguientes líneas en consola:"
    )
    print("  git lfs install")
    print("  git lfs track 'dev/*.pth'")
    print("Asegúrese de persistir el archivo '.gitattributes' generado.")